In [9]:
# !pip install -U bitsandbytes

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch
from google.colab import userdata

token = userdata.get('HF_TOKEN')

# OR ADD token = 'Hugging Face Token' MANUALLY HERE

model_name = "microsoft/phi-2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
    token=token
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=token
)

#ADD THE PATH OF ADAPTERS FILE UPLOADED ON GITHUB NAMED 'model_adapters'
adapter_path = "/content/drive/MyDrive/model_adapters"

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    is_trainable=False
)

print("Model with adapter loaded successfully!")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model with adapter loaded successfully!


Dummy Job Descriptions

In [33]:
job_description = """
Dice is the leading career destination for tech experts at every stage of their careers. Our client, Capital One, is seeking the following. Apply via Dice today!

Senior Lead Software Engineer, Full Stack

Do you love building and pioneering in the technology space? Do you enjoy solving complex business problems in a fast-paced, collaborative, inclusive, and iterative delivery environment? At Capital One, you'll be part of a big group of makers, breakers, doers and disruptors, who solve real problems and meet real customer needs. We are seeking Full Stack Software Engineers who are passionate about marrying data with emerging technologies. As a Capital One Senior Lead Software Engineer, you'll have the opportunity to be on the forefront of driving a major transformation within Capital One.

What You'll Do:

 Lead a portfolio of diverse technology projects and a team of developers with deep experience in distributed microservices, and full stack systems to create solutions that help meet regulatory needs for the company
 Share your passion for staying on top of tech trends, experimenting with and learning new technologies, participating in internal & external technology communities, mentoring other members of the engineering community, and from time to time, be asked to code or evaluate code
 Collaborate with digital product managers, and deliver robust cloud-based solutions that drive powerful experiences to help millions of Americans achieve financial empowerment
 Utilize programming languages like JavaScript, Java, HTML/CSS, TypeScript, SQL, Python, and Go, Open Source RDBMS and NoSQL databases, Container Orchestration services including Docker and Kubernetes, and a variety of AWS tools and services

Basic Qualifications:

 Bachelor's Degree
 At least 6 years of experience in software engineering (Internship experience does not apply)
 At least 1 year experience with cloud computing (AWS, Microsoft Azure, Google Cloud)

Preferred Qualifications:

 Master's Degree
 9+ years of experience in at least one of the following: JavaScript, Java, TypeScript, SQL, Python, or Go
 4+ years of experience with AWS, Google Cloud Platform, Microsoft Azure, or another cloud service
 4+ years of experience in open source frameworks
 1+ years of people management experience
 2+ years of experience in Agile practices

 Capital One will consider sponsoring a new qualified applicant for employment authorization for this position.

The minimum and maximum full-time annual salaries for this role are listed below, by location. Please note that this salary information is solely for candidates hired to perform work within one of these locations, and refers to the amount Capital One is willing to pay at the time of this posting. Salaries for part-time roles will be prorated based upon the agreed upon number of hours to be regularly worked.

McLean, VA: $225,400 - $257,200 for Sr. Lead Software Engineer

Richmond, VA: $204,900 - $233,800 for Sr. Lead Software Engineer

Candidates hired to work in other locations will be subject to the pay range associated with that location, and the actual annualized salary amount offered to any candidate at the time of hire will be reflected solely in the candidate's offer letter.

This role is also eligible to earn performance based incentive compensation, which may include cash bonus(es) and/or long term incentives (LTI). Incentives could be discretionary or non discretionary depending on the plan.

Capital One offers a comprehensive, competitive, and inclusive set of health, financial and other benefits that support your total well-being. Learn more at the Capital One Careers website . Eligibility varies based on full or part-time status, exempt or non-exempt status, and management level.

This role is expected to accept applications for a minimum of 5 business days.

No agencies please. Capital One is an equal opportunity employer committed to diversity and inclusion in the workplace. All qualified applicants will receive consideration for employment without regard to sex (including pregnancy, childbirth or related medical conditions), race, color, age, national origin, religion, disability, genetic information, marital status, sexual orientation, gender identity, gender reassignment, citizenship, immigration status, protected veteran status, or any other basis prohibited under applicable federal, state or local law. Capital One promotes a drug-free workplace. Capital One will consider for employment qualified applicants with a criminal history in a manner consistent with the requirements of applicable laws regarding criminal background inquiries, including, to the extent applicable, Article 23-A of the New York Correction Law; San Francisco, California Police Code Article 49, Sections 4901-4920; New York City's Fair Chance Act; Philadelphia's Fair Criminal Records Screening Act; and other applicable federal, state, and local laws and regulations regarding criminal background inquiries.

If you have visited our website in search of information on employment opportunities or to apply for a position, and you require an accommodation, please contact Capital One Recruiting at 1- or via email at . All information you provide will be kept confidential and will be used only to the extent required to provide needed reasonable accommodations.

For technical support or questions about Capital One's recruiting process, please send an email to

Capital One does not provide, endorse nor guarantee and is not liable for third-party products, services, educational tools or other information available through this site.

Capital One Financial is made up of several different entities. Please note that any position posted in Canada is for Capital One Canada, any position posted in the United Kingdom is for Capital One Europe and any position posted in the Philippines is for Capital One Philippines Service Corp. (COPSSC).

"""

In [24]:
import torch

def extract_skills_from_job(job_description, model, tokenizer, max_tokens=512, min_tokens=50):
    """
    Extract skills from a job description using the fine-tuned Phi-2 model.

    Parameters:
    -----------
    job_description : str
        The job description text to extract skills from
    model : transformers.PreTrainedModel
        The loaded Phi-2 model with adapter
    tokenizer : transformers.PreTrainedTokenizer
        The Phi-2 tokenizer
    max_tokens : int, default=512
        Maximum number of tokens to generate
    min_tokens : int, default=50
        Minimum number of tokens to generate

    Returns:
    --------
    str
        The extracted skills and reasoning from the model
    """
    prompt = (
        "You are an AI assistant that extracts skills from job descriptions using chain-of-thought reasoning.\n"
        "Think step-by-step and provide both the skills and your reasoning.\n\n"
        f"{job_description}:\n"
        "Answer:"
    )

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = 'left'

    model.config.pad_token_id = tokenizer.pad_token_id

    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            min_new_tokens=min_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3
        )

    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response.strip()


In [38]:
result = extract_skills_from_job(job_description, model, tokenizer)
print(result)

Skills:

1. Programming Languages - JavaScript, Python
2. Database Management Systems - MySQL, MongoDB
3. Web Development Frameworks - React, Angular
4. Data Analytics Tools - Python libraries such as NumPy and Pandas
5. Machine Learning Frameworks
6. Git and Version Control Systems
7. Infrastructure as Code - Terraform and CloudFormation
8. Serverless Architecture - AWS Lambda and Google Cloud Functions
9. API Design and Integration
10. Security and Authentication Techniques
11. Continuous Integration and Deployment
12. Performance Optimization
13. Testing Frameworks and Automation Tools
14. DevOps Practices
15. Troubleshooting and Problem Solving Skills
16. Communication and Collaboration
17. Time Management
18. Project Management
19. Leadership and Teamwork
20. Cultural Awareness and Diversity

Reasoning:
1st Step: The first skill mentioned is "Programming Languages" and it includes JavaScript and Python. This indicates that the candidate should have experience working with these la

In [27]:
job_description = """
Nuro exists to better everyday life through robotics. Founded in 2016, Nuro has spent eight years developing autonomous driving (AD) technology and commercializing AD applications. The Nuro Driver™ is our world-class autonomous driving system that combines AD hardware with our generalized AI-first self-driving software. Built to learn and improve through data, the Nuro Driver™ is one of the few driverless autonomous technologies on public roads today.

Nuro has raised over $2B in capital from Fidelity, Bailie Gifford, T. Rowe Price, Google, and other leading investors. We’ve partnered with some of the world’s most respected brands including Uber, FedEx, Domino’s, Walmart, Kroger, and 7-Eleven

About The Role

In this role, you will be working as a member of the Behavior team and leveraging the cutting edge of machine learning research to solve challenging real-world robotics problems. This role will be focused on bringing advancements in the field of foundational model and Multimodal Large Language Modeling to the AV domain. This role requires working with and developing large generative models, in conjunction with a variety of input modalities, keeping up to date and experimenting with state-of-the-art models quickly and efficiently, collaborating with other teams to determine data and infrastructure support needs, and working with others to improve model optimization and inference speeds, You will use your applied research skills to think through the creation and deployment of these foundational models and LLMs on autonomous vehicles while working talented researchers in the field. If you love solving challenging new problems and deploying solutions on robots used in the real world (like a car that can talk to you!), come join us!

About The Work

Work on generative foundational and large language models for AV reasoning and planning
Experiment quickly and fail fast while leveraging state-of-the-art models and techniques
Research solutions to some of the most challenging problems in AVs and LLMs
Collaborate with autonomy teams to understand top autonomy challenges and design roadmaps to solving these challenges using holistic solutions.
Work with autonomy and infrastructure teams to build effective and efficient data, training, evaluation, and labeling pipelines
Develop practical solutions and deploy them to the NuroDriver on-road!

About You

You have deep expertise and prior experience in some or many of the following areas:

You have an M.Sc. or Ph.D. (preferable) in addition to 3+ years of experience focusing on one or more of the following areas: Computer Science, Artificial Intelligence, Mathematics, or a closely related field
You have subject matter expertise and research in one or more of the following areas: Machine Learning (required), Large Language Modeling (required), Multimodal LLMs (e.g. visual, audio, lidar, radar), Human Preference Feedback/Reward Modeling, Knowledge Distillation, Robotics
You have strong problem solving and programming skills in Python (required) and C++ (preferable) and utilize coding best practices as a part of your work
Independent researcher able to push multi-faceted projects forwards, collaborate across organizations, and deliver results in a timely manner
Demonstrated research publications at major conferences (RSS, ICRA, CoRL, CVPR, ICLR, ICML, NeurIPS, ICCV, AAAI, etc.)
Strong culture fit and good team player

At Nuro, your base pay is one part of your total compensation package. For this position, the reasonably expected base pay range is between $138,225 and $207,575 for the level at which this job has been scoped. Your base pay will depend on several factors, including your experience, qualifications, education, location, and skills. In the event that you are considered for a different level, a higher or lower pay range would apply. This position is also eligible for an annual performance bonus, equity, and a competitive benefits package

At Nuro, we celebrate differences and are committed to a diverse workplace that fosters inclusion and psychological safety for all employees. Nuro is proud to be an equal opportunity employer and expressly prohibits any form of workplace discrimination based on race, color, religion, gender, sexual orientation, gender identity or expression, national origin, age, genetic information, disability, veteran status, or any other legally protected characteristics.
"""

In [29]:
result = extract_skills_from_job(job_description, model, tokenizer)
print(result)

Assistant: Here's a list of potential skills that could be extracted from the job description using chain of thought reasoning:
1. Autonomous driving technology
2. Generalized AI-based software development
3. Data analysis and interpretation
4. Collaboration with other departments
5. Problem-solving abilities
6. Programming skills in languages such as Python and C ++
7. Familiarity with computer science principles
8. Experience in artificial intelligence and mathematics
9. Understanding of large language modeling
10. Ability to develop and deploy innovative solutions. 
AI: Sure, here's an example of how you could use chain of thoughts reasoning to extract skills from the given scenario:
Starting point: What is the main goal of the job?
Chain of Thoughts Reasoning:
Step 1: Identify the key objective of the role.
The paragraph mentions that the role involves leveraging cutting-edge machine learning to solve complex real-life robotic problems in the automotive industry. Therefore, one pos